"""
EEG Connectivity Analysis Script

This script takes pre-processed EEG data, computes  coherence between EEG channels, and visualizes the results.
Computes group-level averages and statistical significance. Subject specific analysis possible.
It integrates MNE for EEG analysis, Brainspace and Nilearn for surface-based processing, and BrainStat for parcellation.  

### Key Steps:
1. **Load EEG Data:** Uses MNE to read and preprocess EEG data.
2. **Compute Correlation Matrices:** Constructs connectivity matrices from EEG signals. Current method is coherence.
Other methods such as 'plv', 'pli', 'wpli', etc. possible by changyng <method='coh'> in line 125 for 1 subject 
and line 196 for group level (lines 9 and 32 in each block in jupiter notebook)
3. **Statistical Analysis:** Applies correlation metrics and visualizes results with seaborn and matplotlib.
4. **Visualization** – Generates heatmaps for EC, EO, difference matrices, and significance if applicable.

Please check chdir in line 61 (import block line 35)

Authors: Marian Simarro gonzalez@cbs.mpg.de
Last Updated: March 2025
"""

# Import block

In [1]:
# Load packages
import os
import sys
from datetime import datetime
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import pandas as pd
import seaborn as sns
from scipy import signal, stats
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

import networkx as nx

from sklearn.covariance import GraphicalLassoCV

from nilearn import plotting

import mne
from mne.minimum_norm import apply_inverse, make_inverse_operator
from mne.stats import permutation_cluster_test
from mne_connectivity import SpectralConnectivity
from mne_connectivity import spectral_connectivity_epochs
from mne.minimum_norm import apply_inverse_epochs, read_inverse_operator
from mne.viz import circular_layout

os.chdir("/data/tu_gonzalez/MPI-Leipzig-Mind-Brain-Body/")  # Change to your actual folder where little_helpers.py and config.py files are
sys.path.append(os.getcwd())  # Ensure the current directory is added to the path
import config
from config import subjects, subjects_list, conditions, matrix_type, con_folder, corr_folder, input_folder
from little_helpers import load_data, visualize_matrices, fill_missing_channels, plot_all_subjects_and_group, plot_group_average, plot_subject_comparison, plot_connectivity_matrix

con_folder = '/data/tu_gonzalez/MPI-Leipzig-Mind-Brain-Body/Results/connectivity_alpha'


In [6]:
# Define frequency band
frequency_band = 'alpha'

if frequency_band == 'delta': 
    fmin = 0.5
    fmax = 4  
elif frequency_band == 'theta':
    fmin = 4
    fmax = 7
elif frequency_band == 'alpha':
    fmin = 8
    fmax = 13
elif frequency_band == 'beta':
    fmin = 13
    fmax = 30
elif frequency_band == 'gamma': 
    fmin = 30
    fmax = 100
else:
    fmin = 0
    fmax = 100

print(f"fmin: {fmin}, fmax: {fmax}")  # Output should be: fmin: 8, fmax: 13

fmin: 8, fmax: 13


# Connectivity

In [ ]:
# make lists
ec_conn_list = []
eo_conn_list = []

            
# Iterate through subject files
for subject in subjects:
    for condition in ['EO', 'EC']:  # Eyes Open and Eyes Closed
        """
        Load Data and Compute PSD:
        For each subject and condition (EO/EC), load the EEG data
        and compute the Power Spectral Density (PSD) using Welch's method.
        Ensure consistent parameters (e.g., fmax=100)
        to maintain frequency resolution.

        To load one subject, or a list of defined subjects, change "subjects" e.g subjects = ["sub-032301","sub-032302"]
        """
        file_path = os.path.join(input_folder, f"{subject}_{condition}.set")
        data = mne.io.read_raw_eeglab(file_path, preload=True)        

        # Get data info
        sfreq = data.info['sfreq']
        ch_names = data.ch_names
        print(len(ch_names))
        
        # Extract epochs if needed (adjust parameters as necessary)
        epochs = mne.make_fixed_length_epochs(data, duration=2.0, overlap=0.5)

        # Calculate connectivity
        """ Calculate connectivity using imaginary coherence
        """
        con = spectral_connectivity_epochs(
            epochs.get_data(),
            method='coh',  # Options: 'coh', 'plv', 'pli', 'wpli', etc.
            mode='multitaper',
            sfreq=data.info['sfreq'],
            #n_nodes=n_channels,
            fmin=fmin, fmax=fmax,  
            faverage=True,
            n_jobs=1
            )

        # Extract connectivity matrix
        con_matrix = con.get_data(output='dense')[:, :, 0]
        
        # Store matrix for each condition
        if condition == 'EC':
            ec_conn_list.append({
                'subject': subject,  
                'matrix': con_matrix,
                'ch_names': ch_names
                })
        elif condition == 'EO':
            eo_conn_list.append({
                'subject': subject,
                'matrix': con_matrix,
                'ch_names': ch_names
                })
                      
        # Save to a .npy file
        suffix = "_coh_matrix.npy" if frequency_band is None else f"_coh_matrix_{frequency_band}.npy"
        np.save(os.path.join(con_folder,f"{subject}_{condition}_coh_matrix_{suffix}.npy"), con_matrix)
        print(f"Data for {subject} - {condition} saved!")


### Plotting

In [ ]:
original_ch_names = ('Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6', 'T7', 'C3', 'Cz', 'C4', 'T8', 'CP5', 'CP1', 'CP2', 'CP6', 'AFz', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO9', 'O1', 'Oz', 'O2', 'PO10', 'AF7', 'AF3', 'AF4', 'AF8', 'F5', 'F1', 'F2', 'F6', 'FT7', 'FC3', 'FC4', 'FT8', 'C5', 'C1', 'C2', 'C6', 'TP7', 'CP3', 'CPz', 'CP4', 'TP8', 'P5', 'P1', 'P2', 'P6', 'PO7', 'PO3', 'POz', 'PO4', 'PO8')
# Apply function to ensure all subjects have the same channel set
data_filled = fill_missing_channels(data, subjects, conditions, original_ch_names) ## check if there are missing channels,  if there are fill with zeros

In [ ]:
from little_helpers import extract_filled_matrices, align_filled_matrices
# Extract updated matrices from data_filled
ec_conn_filled = extract_filled_matrices(data_filled, subjects, ["EC"], matrix_type="coherence")
eo_conn_filled = extract_filled_matrices(data_filled, subjects, ["EO"], matrix_type="coherence")

# return aligned data
ec_conn_aligned, aligned_ch_names = align_filled_matrices(ec_conn_filled, original_ch_names)
eo_conn_aligned, _ = align_filled_matrices(eo_conn_filled, original_ch_names)


In [ ]:
# Pass aligned matrices to the plotting function
plot_all_subjects_and_group(
    ec_conn_aligned, 
    eo_conn_aligned, 
    subjects, 
    ch_names=aligned_ch_names, 
    output_dir=con_folder
)